# H3 Toybox｜Colab 简体版（可爱的 MiniMax H3 网页前端）

把想象丢进玩具箱，摇一摇就变成精彩视频 🧸 ComfyUI 只在后台作为引擎运行，**无需打开 ComfyUI 复杂节点界面**，所有直观操作均在 H3 Toybox 网页中完成。

由上往下依次执行。**运行阶段请选 L4 GPU**（24 GB 显存跑 INT8 基础版刚刚好；若需跑 BF16 满血版请使用 BF16 专用笔记本并选 G4 GPU；T4 显存不足无法运行）。

| 步骤 | 说明 |
|---|---|
| ① 检查＋安装 | GPU／驱动不符自动中断；验证通过后自动部署 ComfyUI、3 个核心节点组与 H3 Toybox 简体前端 |
| ② 下载模型 | 约 50 GB（含 7 GB 提示词改写模型，可取消勾选）；勾选「加装生图」再增加约 10 GB |
| ③ 启动＋开网页 | 后台启动 ComfyUI 与 H3 Toybox，通过 Colab 内置代理打开网页；作品默认持久化保存至云端硬盘「H3Toybox」目录 |
| ③-b | （备用）当浏览器登录的 Google 账号与 Colab 账号不一致时，通过 Cloudflare 临时公网地址访问 |

**网页支持的核心功能**
- 🖼 **图片**（可选）：文生图，或上传最多 8 张图片进行微调编辑（换表情、换衣服、换风格、将人物融入场景等）。基于 [P-H3-Image](https://huggingface.co/PrunaAI/P-H3-Image-Comfy) 复用同一主模型，仅多约 10 GB
- 🤖 **短片 Agent**：输入一句话自动规划拆解为多段分镜头短片（需先在 ⚙ 设置中填写 Gemini API Key）
- 🧸 **多模态参考生成**：支持最多上传 9 张图片／3 段视频／3 段音频作为参考，提示词中通过 `@图片1` 等格式精准锚定
- 🎞 **首尾帧控制**：支持仅首帧、仅尾帧、或首尾双帧过渡生成
- ✏️ **纯文字生成**：仅输入提示词生成完整动态视频
- 🧵 **视频无缝延伸**：参考视频勾选「延伸这支视频」，新片段将从原视频最后一帧平滑过渡并自动拼合成长视频
- ✨ **智能提示词改写**：将简短中文描述自动扩展为符合 H3 规范的高质量英文提示词（改写结果默认收起）。预设内置 H3 官方 9B 本地改写模型；亦可在 ⚙ 设置中无缝切换为 **Gemini** 或 **OpenAI 兼容接口**
- ⚡ **快速 Turbo 模式**（默认开启）：8 步极速出片；关闭后为 12 步高精出片
- 📁 **素材库管理**：支持将图片、视频、音频拖曳至文件夹分类整理
- 实时生成进度展示：精准显示当前步数与预计剩余时间

**Hugging Face 访问令牌（强烈建议配置）**：在 Colab 左侧栏「🔑 密钥」中添加 `HF_TOKEN`（建议使用**只读 (Read)** 令牌），并开启「笔记本访问权限」。

模型：`Minimax-h3_Singularity_x_10Eros-Max_PRISM`（int8）＋官方 Qwen3-VL 文本编码器与 VAE；辅助 LoRA、语义桥、潜空间放大器与改写模型镜像托管于 [dseditor/minimax-h3-studio-assets](https://huggingface.co/dseditor/minimax-h3-studio-assets)（各文件来源详见该页面）。模型授权依照各开源原版协议执行。

- 简体汉化版仓库：<https://github.com/mxllAs/minimax-h3-colab-zh>
- 原作者上游仓库：<https://github.com/dseditor/minimax-h3-colab>

In [ ]:
#@title ① 检查环境＋安装 ComfyUI、核心节点与 H3 Toybox（GPU 或驱动不满足要求将自动终止，不会安装任何组件）
import torch, shutil, subprocess, sys, os
from pathlib import Path
if not torch.cuda.is_available():
    raise RuntimeError('未检测到 GPU：请到「代码执行程序 → 更改运行时类型」选择 L4 GPU 后重新运行本单元格')
cap = torch.cuda.get_device_capability(0)
drv = subprocess.run(['nvidia-smi', '--query-gpu=driver_version', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
print('GPU:', torch.cuda.get_device_name(0), '| 计算能力:', cap, '| 显存:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), 'GiB')
print('驱动版本:', drv, '| 可用磁盘空间:', round(shutil.disk_usage('/content').free / 2**30, 1), 'GiB')
if cap < (8, 0):
    raise RuntimeError(f'当前显卡算力架构（计算能力 {cap}）不支持运行 int8 模型，未安装任何组件。请更换为 L4（或 G4 / A100）实例后重试。')
if int(drv.split('.')[0]) < 580:
    raise RuntimeError(f'当前驱动版本 {drv} 不支持 CUDA 13（需 580 以上），未安装任何组件。请重新分配一台运行时（L4）后再试。')
if shutil.disk_usage('/content').free < 72 * 2**30:
    raise RuntimeError('可用磁盘空间不足 72 GiB，无法容纳约 60 GB 的模型文件。请重新分配运行时实例。')
print('✅ 环境检查通过，开始安装依赖环境')
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN'); print('已成功读取 HF_TOKEN')
except Exception:
    print('未设置 HF_TOKEN，后续将使用匿名模式下载（高峰期可能会遇到限速）')

def clone(url, dest):
    if not Path(dest).exists():
        subprocess.run(['git', 'clone', '--depth', '1', url, str(dest)], check=True)
    else:
        subprocess.run(['git', '-C', str(dest), 'pull', '--ff-only'], check=False)

COMFY = Path('/content/ComfyUI')
clone('https://github.com/Comfy-Org/ComfyUI.git', COMFY)
if not (torch.version.cuda or '').startswith('13'):
    print('正在安装 CUDA 13 专版 PyTorch（int8 ConvRot 权重所需）…')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'torchvision', 'torchaudio',
                    '--index-url', 'https://download.pytorch.org/whl/cu130'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(COMFY / 'requirements.txt'), 'huggingface_hub', 'hf_xet'], check=True)
# 工作流用到的三个核心节点组（均不会改动 PyTorch 版本）
NODES = {
    'minimax-h3-audio-t8': 'https://github.com/T8mars/comfyui-minimax-h3-audio-T8',        # 语义桥 Semantic Bridge
    'comfyui-SelfLift': 'https://github.com/facok/comfyui-SelfLift',                      # 低分辨率→潜空间放大→高分辨率
    'ComfyUI-H3-Motion-Context': 'https://github.com/NikoDemon80/ComfyUI-H3-Motion-Context',  # 视频延伸扩展（气锁）
}
for name, url in NODES.items():
    clone(url, COMFY / 'custom_nodes' / name)
# 导演 Skill 仅为文本提示词模板：保存在 custom_nodes 外部，不作为节点加载
clone('https://github.com/T8mars/comfyui-minimax-h3-prompt-enhancer-T8', '/content/t8-enhancer')
# H3 Toybox 简体中文前端本体
clone('https://github.com/mxllAs/minimax-h3-colab-zh.git', '/content/minimax-h3-colab')
APP = Path('/content/minimax-h3-colab/toybox')
print('ffmpeg 状态:', shutil.which('ffmpeg') or '未检测到（视频延伸功能将不可用）')
out = subprocess.run([sys.executable, '-c', 'import torch;print(torch.__version__, torch.version.cuda, torch.cuda.is_available())'],
                     capture_output=True, text=True)
print('ComfyUI 调用的 PyTorch 环境：', out.stdout.strip(), out.stderr[-300:])
print('✅ 环境安装完成')

In [ ]:
#@title ② 下载模型（约 50 GB）
# 默认直接从 Hugging Face 下载（速度最快）。HF 访问令牌已在步骤 ① 中读取。
# 仅当 Hugging Face 遇到网络波动或流量配额限制时，才勾选 USE_DRIVE_CACHE：
#   勾选后会优先从「我的云端硬盘/H3_models」读取缓存（速度稍慢）；云端硬盘中不存在的文件才会去 HF 下载并顺带同步一份缓存。
USE_DRIVE_CACHE = False  #@param {type:"boolean"}
#@markdown 不勾选可少下载 7 GB：网页的本地提示词改写功能将被关闭，可在 ⚙ 设置中切换为 Gemini 或 OpenAI 兼容 API。
INCLUDE_REWRITER = True  #@param {type:"boolean"}
#@markdown 🖼 启用生图功能（需额外下载约 10 GB）：开启后可在网页「图片」选项卡中文生图、最多支持 8 张图片微调编辑。直接复用同一主干 H3 模型。
INSTALL_IMAGE = True  #@param {type:"boolean"}
from huggingface_hub import hf_hub_download
import shutil, time
CACHE = None
if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE = Path('/content/drive/MyDrive/H3_models'); CACHE.mkdir(parents=True, exist_ok=True)
ASSETS = 'dseditor/minimax-h3-studio-assets'
files = [  # (repo, 文件名, 存放在 models/ 下的目标子目录)
    ('OliviaRossi/Minimax-h3_Singularity-x-10Eros-Max_PRISM', 'Minimax-h3_Singularity_x_10Eros-Max_PRISM_int8.safetensors', 'diffusion_models'),
    ('Comfy-Org/MiniMax-H3', 'text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors', 'text_encoders'),
    ('Comfy-Org/MiniMax-H3', 'vae/minimax_h3_video_vae_int8_convrot.safetensors', 'vae'),
    ('Comfy-Org/MiniMax-H3', 'vae/minimax_h3_audio_vae_fp32.safetensors', 'vae'),
    # pika flex Ref2VA（2026-11-04 取代 dmad）：视频与 32×32 TTS 共用该权重
    ('Comfy-Org/MiniMax-H3', 'loras/pika_h3_flex_ref2va_v1_bf16.safetensors', 'loras'),
] + [(ASSETS, f, str(Path(f).parent)) for f in [
    'loras/minimax_h3_turbo_v4_step600_ema_DasiwaREF2VAHybridV1_0_curveproj1025_compat_v001.safetensors',
    'loras/MysticXXX_MMH3-V2.safetensors',
    'loras/H3_Combat_V2.safetensors',
    'loras/Motion_Repair.safetensors',
    'loras/H3_SPEED_slider.safetensors',
    'semantic_bridge/t8_compat/BUNNY_H3_ActionLogic_Bridge_V1_T8_Compat.safetensors',
    'latent_upscale_models/minimax_h3_latent_upscaler_3d_fp16_V2.safetensors',
] + (['text_encoders/minimax_h3_rewriter_ref2va_qwen3.5_9b_w4a8.safetensors'] if INCLUDE_REWRITER else [])] + ([
    ('PrunaAI/P-H3-Image-Comfy', 'loras/p-h3-image_lora_bf16.safetensors', 'loras'),
    ('PrunaAI/P-H3-Image-Comfy', 'text_encoders/p-h3-image_text_encoder_int8_convrot.safetensors', 'text_encoders'),
    ('PrunaAI/P-H3-Image-Comfy', 'vae/p-h3-image_vae_int8_convrot.safetensors', 'vae'),
] if INSTALL_IMAGE else [])
t0 = time.time()
for repo, name, folder in files:
    target = COMFY / 'models' / folder / Path(name).name
    target.parent.mkdir(parents=True, exist_ok=True)
    cached = CACHE / Path(name).name if CACHE else None
    if not target.is_file():
        if cached and cached.is_file():
            print('从云端硬盘读取缓存', cached.name); shutil.copy(str(cached), str(target))
        else:
            got = Path(hf_hub_download(repo_id=repo, filename=name, local_dir='/content/hf'))
            shutil.move(str(got), str(target))
    if cached and not cached.is_file():
        print('缓存副本已写入云端硬盘', cached.name); shutil.copy(str(target), str(cached))
    print(f'{folder:<28} {target.name[:60]:<60} {target.stat().st_size / 2**30:5.1f} GiB')
# 官方提示词编写指南（本地改写模型遵循此规范）
GUIDES = APP / 'guides'; GUIDES.mkdir(exist_ok=True)
for g in ('VIDEO_PROMPT_WRITING_GUIDE_ref_en.md', 'VIDEO_PROMPT_WRITING_GUIDE_base_en.md'):
    got = Path(hf_hub_download(repo_id='MiniMaxAI/MiniMax-H3', filename='docs/' + g, local_dir='/content/hf'))
    shutil.copy(str(got), str(GUIDES / g))
print(f'✅ 下载完成，耗时 {(time.time() - t0) / 60:.1f} 分钟')

In [ ]:
#@title ③ 启动 ComfyUI（后台）＋ H3 Toybox 并打开网页
# 运行后下方将生成访问链接；若显示为 localhost 也无需担心，点击即可在新标签页中打开。
# ⚠️ 打开标签页的浏览器必须登录与当前 Colab 相同的 Google 账号；若账号不一致请改用下方备用单元格 ③-b。
#@markdown **作品自动同步至 Google 云端硬盘**：素材库、生成目录、产出视频及生成记录均存放在「我的云端硬盘/H3Toybox」，
#@markdown 即使实例意外断开或重置，重新运行 ①②③ 即可自动恢复历史作品。首次运行会弹出云端硬盘授权弹窗，点击同意即可。
SAVE_TO_DRIVE = True  #@param {type:"boolean"}
#@markdown 将 API 密钥持久化保存至云端硬盘（下次免重新输入；但云端硬盘中将生成密钥配置文件）
SAVE_API_KEYS = False  #@param {type:"boolean"}
import subprocess, time, urllib.request, json, sys, os, shutil
def wait_http(url, name, log, tries=180):
    for i in range(tries):
        try:
            urllib.request.urlopen(url, timeout=2); print(name, '已成功启动'); return
        except Exception:
            time.sleep(2)
    print(open(log, encoding='utf-8', errors='ignore').read()[-3000:])
    raise RuntimeError(name + ' 启动失败，请检查上方日志输出')

for proc in ('server', 'toybox'):
    if proc in globals() and globals()[proc].poll() is None:
        globals()[proc].terminate(); globals()[proc].wait()
# 作品存放路径：Google 云端硬盘（授权成功）或当前运行时的临时存储空间
DATA = Path('/content/h3toybox_data')
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DATA = Path('/content/drive/MyDrive/H3Toybox')
    except Exception as e:
        print('⚠️ 未能成功挂载 Google 云端硬盘（', e, '）：本次作品仅保存在临时空间中，实例断开后将被清空')
DATA.mkdir(parents=True, exist_ok=True)
# ComfyUI 的输出目录直接设在 DATA/output。不能用软链接将 output/H3Toybox 链接出去：
# ComfyUI 保存与 /view 接口会解析符号链接并校验真实物理路径，超出 output 范围将报错：
# 「Saving image outside the output folder is not allowed」。
output_root = DATA / 'output'
videos, out_dir = output_root / 'H3Toybox', COMFY / 'output' / 'H3Toybox'
videos.mkdir(parents=True, exist_ok=True)
# 迁移旧版残留文件，确保历史生成的视频正常展示：DATA/videos、旧符号链接指向的目录、本机 output/H3Toybox
old_dirs = [DATA / 'videos']
if out_dir.is_symlink():
    old_dirs.append(out_dir.resolve())
    out_dir.unlink()
elif out_dir.is_dir():
    old_dirs.append(out_dir)
for old in old_dirs:
    if old.is_dir() and old.resolve() != videos.resolve():
        for f in old.iterdir():
            if f.is_file() and not (videos / f.name).exists():
                shutil.move(str(f), str(videos / f.name))
# ComfyUI 仅作为推理后台：仅监听 127.0.0.1，不对外暴露管理界面
server = subprocess.Popen([sys.executable, 'main.py', '--listen', '127.0.0.1', '--port', '8188',
                           '--output-directory', str(output_root)],
                          cwd=str(COMFY), stdout=open('/content/comfyui.log', 'w'), stderr=subprocess.STDOUT)
wait_http('http://127.0.0.1:8188/system_stats', 'ComfyUI', '/content/comfyui.log')
print('🎬 视频输出路径：', videos)
# H3 Toybox 的 Colab 配置（保留用户在网页端修改的设置项，仅覆盖当前实例的环境路径）
(COMFY / 'input').mkdir(exist_ok=True)
cfg_path = DATA / 'config.json'
cfg = json.loads(cfg_path.read_text(encoding='utf-8')) if cfg_path.exists() else {}
cfg.update({
    'comfy_url': 'http://127.0.0.1:8188',
    'comfy_input_dir': str(COMFY / 'input'),
    'skills_dir': '/content/t8-enhancer/directional_skills',
    'free_after_rewrite': True,   # Colab 显存较小：提示词改写完成后立即卸载 9B 模型
})
cfg.pop('guides', None)
# 🖼 图片选项卡：Colab 环境默认使用 P-H3-Image（复用主干模型）；检查权重文件是否存在
PH3 = ('loras/p-h3-image_lora_bf16.safetensors', 'text_encoders/p-h3-image_text_encoder_int8_convrot.safetensors',
       'vae/p-h3-image_vae_int8_convrot.safetensors')
# 🤖 短片 Agent：Colab 上没有本地大语言模型，默认改用 Gemini（可在网页 ⚙ 设置中填写 API Key）
cfg['agent'] = dict(cfg.get('agent') or {}, provider=(cfg.get('agent') or {}).get('provider', 'gemini'))
cfg['image'] = dict(cfg.get('image') or {}, engine='ph3',
                    installed=all((COMFY / 'models' / f).is_file() for f in PH3))
print('🖼 生图模块：', '已就绪' if cfg['image']['installed'] else '未安装（需在步骤 ② 中勾选 INSTALL_IMAGE）')
cfg_path.write_text(json.dumps(cfg, ensure_ascii=False, indent=1), encoding='utf-8')
secrets = DATA / 'secrets.json' if SAVE_API_KEYS else Path('/content/h3toybox_secrets.json')
toybox = subprocess.Popen([sys.executable, 'server.py'], cwd=str(APP),
                          env=dict(os.environ, H3TOYBOX_PORT='8190', H3TOYBOX_DATA=str(DATA), H3TOYBOX_SECRETS=str(secrets)),
                          stdout=open('/content/toybox.log', 'w'), stderr=subprocess.STDOUT)
wait_http('http://127.0.0.1:8190/api/config', 'H3 Toybox', '/content/toybox.log', tries=30)
print('📁 作品存放目录：', DATA)
from google.colab import output
output.serve_kernel_port_as_window(8190, anchor_text='🧸 点击打开 H3 Toybox 网页')

In [ ]:
#@title ③-b（备用）通过 Cloudflare 隧道打开 H3 Toybox：浏览器与 Colab 账号不一致时使用
# ⚠️ 该临时网址无密码保护，获取链接者均可调用你的 GPU 算力 —— 请切勿分享至公开渠道。
import subprocess, re, time
if not Path('/content/cloudflared').exists():
    subprocess.run(['wget', '-q', '-O', '/content/cloudflared',
                    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'], check=True)
    subprocess.run(['chmod', '+x', '/content/cloudflared'], check=True)
# 隧道输出直接重定向至日志文件，切勿使用 PIPE 管道读取：管道无消费者时会导致缓冲区阻塞挂起
if 'tun' in globals() and tun.poll() is None:
    tun.terminate(); tun.wait()
open('/content/cloudflared.log', 'w').close()
tun = subprocess.Popen(['/content/cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8190'],
                       stdout=open('/content/cloudflared.log', 'a'), stderr=subprocess.STDOUT)
url = None
for i in range(60):
    time.sleep(1)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/cloudflared.log').read())
    if m:
        url = m.group(0); break
print('👉 H3 Toybox 访问地址：', url or '（未能成功获取公网网址，请查看 /content/cloudflared.log）')

In [ ]:
#@title 🔍 故障排查：查看 GPU 状态、ComfyUI 与 H3 Toybox 运行日志
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
print(subprocess.run(['free', '-g'], capture_output=True, text=True).stdout)
log = open('/content/comfyui.log', encoding='utf-8', errors='ignore').read()
keys = ('error', 'Error', 'Traceback', 'out of memory', 'SelfLift timing] high_resolution total', 'Prompt executed', 'IMPORT FAILED')
print('\n'.join(l for l in log.splitlines() if any(k in l for k in keys))[-4000:])
print('---- H3 Toybox 运行日志 ----')
print(open('/content/toybox.log', encoding='utf-8', errors='ignore').read()[-2000:])